# 06 Spell Out Numbers

Reads `processed/05_lemmatized.csv` and writes numbers as words, so "4 legs" and "four legs" end up as the same thing.

| Written as | Becomes |
|---|---|
| `4`, `1,000`, `5280` | four, one thousand, five thousand two hundred eighty |
| `2011`, `1900` (years) | twenty eleven, nineteen hundred |
| `1st`, `2nd`, `25th` | first, second, twenty fifth |
| `3.5` | three point five |
| `9/11`, `20/20` | nine eleven, twenty twenty |
| `1/4`, `3/4` | one quarter, three quarters |
| `50s`, `1950s` | fifties, nineteen fifties |

Hyphens and "and" are removed, so every number is a list of plain words. Mixed letter-and-digit names (`h2o`, `co2`, `3d`, `ww2`) are left as written.

Output: `processed/06_numbers.csv`, and `processed/06_number_changes.csv` listing every number and what it became.

In [1]:
import pandas as pd
import re
from collections import Counter
from num2words import num2words

In [2]:
data = pd.read_csv("processed/05_lemmatized.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} rows")

185,824 rows


In [3]:
FRACTIONS = {"1/2": "one half", "1/3": "one third", "2/3": "two thirds", "1/4": "one quarter", "3/4": "three quarters"}

def words(n, **kwargs):
    text = num2words(n, **kwargs).replace("-", " ").replace(",", "")
    return " ".join(w for w in text.split() if w != "and")

def spell(token):
    """Returns the number written as words, or None if the token is not a plain number."""
    t = token.lower()
    if t in FRACTIONS:
        return FRACTIONS[t]
    if re.fullmatch(r"\d{1,3}(,\d{3})+|\d+", t):                 # 4, 1,000, 5280
        n = int(t.replace(",", ""))
        if "," not in t and 1100 <= n <= 2099 and n % 1000 != 0:   # years
            return words(n, to="year")
        return words(n)
    m = re.fullmatch(r"(\d+)(st|nd|rd|th)", t)                      # 1st, 25th
    if m:
        return words(int(m.group(1)), to="ordinal")
    if re.fullmatch(r"\d+\.\d+", t):                                # 3.5
        whole, frac = t.split(".")
        return words(int(whole)) + " point " + " ".join(words(int(d)) for d in frac)
    m = re.fullmatch(r"(\d+)/(\d+)", t)                            # 9/11, 20/20
    if m:
        return words(int(m.group(1))) + " " + words(int(m.group(2)))
    m = re.fullmatch(r"(\d{2}|\d{4})s", t)                         # 50s, 1950s
    if m:
        return decade(int(m.group(1)))
    return None

def decade(n):
    text = words(n, to="year") if n >= 1000 else words(n)
    return re.sub(r"y$", "ies", text) if n % 100 else text + 's'

In [4]:
print(spell("4"), "|", spell("1,000"), "|", spell("5280"), "|", spell("2011"), "|", spell("1900"), "|", spell("2000"))
print(spell("1st"), "|", spell("25th"), "|", spell("3.5"), "|", spell("9/11"), "|", spell("1/4"), "|", spell("50s"), "|", spell("1950s"))
print(spell("h2o"), spell("3d"))

four | one thousand | five thousand two hundred eighty | twenty eleven | nineteen hundred | two thousand
first | twenty fifth | three point five | nine eleven | one quarter | fifties | nineteen fifties
None None


In [5]:
changes = Counter()
new_lemmas, new_tokens = [], []

for tokens in data["tokens"]:
    out = []
    for item in tokens.split():
        word, lemma, pos = item.rsplit("|", 2)
        spelled = spell(word)
        if spelled is None:
            out.append((word, lemma, pos))
        else:
            changes[(word, spelled)] += 1
            out.extend((word, w, "NUM") for w in spelled.split())
    new_tokens.append(" ".join("|".join(t) for t in out))
    new_lemmas.append(" ".join(t[1] for t in out))

data["tokens"] = new_tokens
data["answer_lemmatized"] = new_lemmas
print(f"{sum(changes.values()):,} numbers spelled out ({len(changes)} different)")

3,229 numbers spelled out (264 different)


In [6]:
log = pd.DataFrame([(a, b, n) for (a, b), n in changes.items()], columns=["number", "words", "count"])
log = log.sort_values("count", ascending=False)
log.to_csv("processed/06_number_changes.csv", index=False)
log.head(30)

,number,words,count
14,2,two,315
16,4,four,193
33,3,three,168
10,5,five,163
28,1,one,158
58,12,twelve,131
4,10,ten,125
54,24,twenty four,122
21,7,seven,119
65,60,sixty,101


In [7]:
# anything with a digit still left
left = Counter()
for tokens in data["tokens"]:
    for item in tokens.split():
        word = item.rsplit("|", 2)[1]   # the lemma
        if re.search(r"\d", word):
            left[word] += 1
print(f"{sum(left.values())} left as written:", left.most_common(25))

279 left as written: [('h2o', 22), ('3d', 22), ('co2', 7), ('o2', 6), ('ww2', 5), ('2d', 5), ('12:00', 4), ('mp3', 4), ('h20', 4), ('o3', 3), ('y2k', 3), ('v8', 3), ('k-12', 3), ('4x4', 3), ('1,2,3', 3), ('mc2', 3), ('2,4,6', 3), ('401k', 2), ('4x', 2), ('omega-3', 2), ('2pac', 2), ('u2', 2), ('catch-22', 2), ('mc^2', 2), ('.25', 2)]


In [8]:
out = data[["row_id", "subject_id", "cue", "answer_lemmatized", "tokens"]]
out.to_csv("processed/06_numbers.csv", index=False)
print(f"Saved {len(out):,} rows to processed/06_numbers.csv")

Saved 185,824 rows to processed/06_numbers.csv
